In [1]:
import io
import zipfile
import requests
import pandas as pd

# Official UCI download
url = "https://archive.ics.uci.edu/static/public/352/online%2Bretail.zip"

response = requests.get(url, timeout=60)
response.raise_for_status()

with zipfile.ZipFile(io.BytesIO(response.content)) as archive:
    with archive.open("Online Retail.xlsx") as file:
        df = pd.read_excel(file, engine="openpyxl")

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (541909, 8)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [2]:
print("COLUMN INFORMATION")
df.info()

print("\nMISSING VALUES")
print(df.isnull().sum())

print("\nDUPLICATED ROWS")
print(df.duplicated().sum())

print("\nDATE RANGE")
print("Start:", df["InvoiceDate"].min())
print("End:", df["InvoiceDate"].max())

print("\nCANCELLED TRANSACTIONS")
cancelled = df["InvoiceNo"].astype(str).str.startswith("C")
print(cancelled.sum())

print("\nINVALID OR RETURNED QUANTITIES")
print((df["Quantity"] <= 0).sum())

print("\nZERO OR NEGATIVE PRICES")
print((df["UnitPrice"] <= 0).sum())

print("\nNUMBER OF COUNTRIES")
print(df["Country"].nunique())

COLUMN INFORMATION
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[ns]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 33.1+ MB

MISSING VALUES
InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

DUPLICATED ROWS
5268

DATE RANGE
Start: 2010-12-01 08:26:00
End: 2011-12-09 12:50:0

Step 2: Clean and prepare the retail data

Valid sales analysis
Customer analysis
Returns and cancellations
Cleaning audit report

In [3]:
import pandas as pd
import numpy as np

# Preserve the original dataset
raw_df = df.copy()

print("Original shape:", raw_df.shape)

Original shape: (541909, 8)


2.1 Standardise data types

In [4]:
df = raw_df.copy()

df["InvoiceNo"] = df["InvoiceNo"].astype(str).str.strip()
df["StockCode"] = df["StockCode"].astype(str).str.strip()
df["Description"] = df["Description"].astype("string").str.strip()
df["Country"] = df["Country"].astype("string").str.strip()

df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce")
df["UnitPrice"] = pd.to_numeric(df["UnitPrice"], errors="coerce")
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")

# Nullable integer preserves missing CustomerID values
df["CustomerID"] = pd.to_numeric(
    df["CustomerID"], errors="coerce"
).astype("Int64")

df.dtypes


,0
InvoiceNo,object
StockCode,object
Description,string[python]
Quantity,int64
InvoiceDate,datetime64[ns]
UnitPrice,float64
CustomerID,Int64
Country,string[python]


2.2 Create quality-control flags

In [5]:
df["IsCancellation"] = (
    df["InvoiceNo"]
    .str.upper()
    .str.startswith("C")
)

df["IsReturn"] = df["Quantity"] < 0
df["InvalidQuantity"] = df["Quantity"] <= 0
df["InvalidPrice"] = df["UnitPrice"] <= 0
df["MissingCustomer"] = df["CustomerID"].isna()
df["MissingDescription"] = df["Description"].isna()
df["IsDuplicate"] = df.duplicated(keep="first")

2.3 Produce an audit report  

In [6]:
audit_report = pd.DataFrame({
    "Quality issue": [
        "Original rows",
        "Duplicate rows",
        "Cancelled invoices",
        "Negative quantities/returns",
        "Zero or negative quantities",
        "Zero or negative prices",
        "Missing CustomerID",
        "Missing description",
        "Invalid invoice dates"
    ],
    "Number of rows": [
        len(df),
        df["IsDuplicate"].sum(),
        df["IsCancellation"].sum(),
        df["IsReturn"].sum(),
        df["InvalidQuantity"].sum(),
        df["InvalidPrice"].sum(),
        df["MissingCustomer"].sum(),
        df["MissingDescription"].sum(),
        df["InvoiceDate"].isna().sum()
    ]
})

audit_report["Percentage"] = (
    audit_report["Number of rows"] / len(df) * 100
).round(2)

audit_report

,Quality issue,Number of rows,Percentage
0,Original rows,541909,100.00
1,Duplicate rows,5268,0.97
2,Cancelled invoices,9288,1.71
3,Negative quantities/returns,10624,1.96
4,Zero or negative quantities,10624,1.96
5,Zero or negative prices,2517,0.46
6,Missing CustomerID,135080,24.93
7,Missing description,1454,0.27
8,Invalid invoice dates,0,0.00


2.4 Separate returns and cancellations

Do not simply delete these records permanently. Returns are commercially useful because they can reveal problematic products or unusual customer behaviour.

In [7]:
returns_df = df[
    df["IsCancellation"] |
    df["IsReturn"]
].copy()

print("Returns/cancellations:", returns_df.shape)

Returns/cancellations: (10624, 15)


In [14]:
# Confirm complete raw dataset
print("Raw shape:", raw_df.shape)
print("Date range:")
print(raw_df["InvoiceDate"].min(), raw_df["InvoiceDate"].max())

assert raw_df.shape[0] == 541909

Raw shape: (541909, 8)
Date range:
2010-12-01 08:26:00 2011-12-09 12:50:00


In [15]:
cleaning_base = raw_df.copy()

# Correct data types
cleaning_base["InvoiceNo"] = (
    cleaning_base["InvoiceNo"].astype(str).str.strip()
)

cleaning_base["StockCode"] = (
    cleaning_base["StockCode"].astype(str).str.strip()
)

cleaning_base["Description"] = (
    cleaning_base["Description"].astype("string").str.strip()
)

cleaning_base["Country"] = (
    cleaning_base["Country"].astype("string").str.strip()
)

cleaning_base["InvoiceDate"] = pd.to_datetime(
    cleaning_base["InvoiceDate"],
    errors="coerce"
)

cleaning_base["CustomerID"] = pd.to_numeric(
    cleaning_base["CustomerID"],
    errors="coerce"
).astype("Int64")

# Remove exact duplicates
cleaning_base = cleaning_base.drop_duplicates().copy()

# Preserve returns separately
returns_df = cleaning_base[
    cleaning_base["InvoiceNo"].str.upper().str.startswith("C") |
    (cleaning_base["Quantity"] <= 0)
].copy()

# Create valid sales dataset
sales_clean = cleaning_base[
    ~cleaning_base["InvoiceNo"].str.upper().str.startswith("C") &
    (cleaning_base["Quantity"] > 0) &
    (cleaning_base["UnitPrice"] > 0) &
    cleaning_base["Description"].notna() &
    cleaning_base["InvoiceDate"].notna()
].copy()

sales_clean["Revenue"] = (
    sales_clean["Quantity"] * sales_clean["UnitPrice"]
).round(2)

sales_clean["YearMonth"] = (
    sales_clean["InvoiceDate"].dt.to_period("M").astype(str)
)

sales_clean["DayName"] = sales_clean["InvoiceDate"].dt.day_name()
sales_clean["Hour"] = sales_clean["InvoiceDate"].dt.hour

# Customer dataset must be smaller than sales dataset
customer_sales = sales_clean.dropna(
    subset=["CustomerID"]
).copy()

customer_sales["CustomerID"] = (
    customer_sales["CustomerID"].astype(int)
)

print("Sales:", sales_clean.shape)
print("Customer sales:", customer_sales.shape)
print("Returns:", returns_df.shape)
print("Final date:", sales_clean["InvoiceDate"].max())

assert len(customer_sales) <= len(sales_clean)
assert sales_clean["InvoiceDate"].max() >= pd.Timestamp("2011-12-09")

Sales: (524878, 12)
Customer sales: (392692, 12)
Returns: (10587, 8)
Final date: 2011-12-09 12:50:00


2.5 Create the clean sales dataset

In [8]:
sales_clean = df.copy()

# Remove exact duplicates
sales_clean = sales_clean.drop_duplicates()

# Keep genuine completed sales
sales_clean = sales_clean[
    (~sales_clean["IsCancellation"]) &
    (sales_clean["Quantity"] > 0) &
    (sales_clean["UnitPrice"] > 0) &
    (sales_clean["Description"].notna()) &
    (sales_clean["InvoiceDate"].notna())
].copy()

# Calculate transaction-line revenue
sales_clean["Revenue"] = (
    sales_clean["Quantity"] * sales_clean["UnitPrice"]
).round(2)

print("Clean sales shape:", sales_clean.shape)
sales_clean.head()

Clean sales shape: (529720, 16)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,IsCancellation,IsReturn,InvalidQuantity,InvalidPrice,MissingCustomer,MissingDescription,IsDuplicate,Revenue
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,False,False,False,False,False,False,False,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,False,False,False,False,False,False,False,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,False,False,False,False,False,False,False,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,False,False,False,False,False,False,False,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,False,False,False,False,False,False,False,20.34


2.6 Add useful date features

In [9]:
sales_clean["Date"] = sales_clean["InvoiceDate"].dt.date
sales_clean["Year"] = sales_clean["InvoiceDate"].dt.year
sales_clean["Month"] = sales_clean["InvoiceDate"].dt.month
sales_clean["MonthName"] = sales_clean["InvoiceDate"].dt.month_name()
sales_clean["YearMonth"] = sales_clean["InvoiceDate"].dt.to_period("M").astype(str)
sales_clean["Day"] = sales_clean["InvoiceDate"].dt.day
sales_clean["DayName"] = sales_clean["InvoiceDate"].dt.day_name()
sales_clean["Hour"] = sales_clean["InvoiceDate"].dt.hour
sales_clean["Quarter"] = sales_clean["InvoiceDate"].dt.quarter
sales_clean["WeekOfYear"] = sales_clean["InvoiceDate"].dt.isocalendar().week.astype(int)

2.7 Create a customer-level dataset

Missing customer IDs can still contribute to overall sales totals, but they cannot be used for customer segmentation.

In [10]:
customer_sales = sales_clean.dropna(
    subset=["CustomerID"]
).copy()

customer_sales["CustomerID"] = (
    customer_sales["CustomerID"].astype(int)
)

print("All valid sales:", len(sales_clean))
print("Sales with identified customers:", len(customer_sales))

All valid sales: 529720
Sales with identified customers: 397501


2.8 Validate the results

In [11]:
assert sales_clean.duplicated().sum() == 0
assert sales_clean["InvoiceDate"].notna().all()
assert (sales_clean["Quantity"] > 0).all()
assert (sales_clean["UnitPrice"] > 0).all()
assert (sales_clean["Revenue"] >= 0).all()

print("All cleaning validation checks passed.")

All cleaning validation checks passed.


In [13]:
# Confirm complete raw dataset
print("Raw shape:", raw_df.shape)
print("Date range:")
print(raw_df["InvoiceDate"].min(), raw_df["InvoiceDate"].max())

assert raw_df.shape[0] == 541909

Raw shape: (541909, 8)
Date range:
2010-12-01 08:26:00 2011-12-09 12:50:00


2.9 Save the processed datasets

In [12]:
sales_clean.to_csv(
    "online_retail_clean.csv",
    index=False
)

customer_sales.to_csv(
    "online_retail_customer_analysis.csv",
    index=False
)

returns_df.to_csv(
    "online_retail_returns.csv",
    index=False
)

audit_report.to_csv(
    "data_cleaning_audit.csv",
    index=False
)

print("All processed files saved successfully.")

All processed files saved successfully.
